#1. Project Folder Structure

In [ ]:
import os

folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "dashboard",
    "reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folder structure created successfully!")

Project folder structure created successfully!


#2. Dependencies and Requirements

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly sqlalchemy requests scipy

In [ ]:
requirements = """pandas
numpy
matplotlib
seaborn
plotly
sqlalchemy
requests
scipy
jupyter
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


#3. Load and Inspect the 10 Raw Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
raw_path = "/content/drive/MyDrive/Bluestock /capstone/raw"

In [ ]:
import os

print(os.listdir(raw_path))

['1788499980509-304c1255-08_investor_transactions.csv', '1788499982117-e3d6ab98-09_portfolio_holdings.csv', '1788499982615-f9647ab2-10_benchmark_indices.csv', '1788499983024-b042c300-01_fund_master.csv', '1788499983331-4389156d-02_nav_history.csv', '1788499984405-d702a6c6-04_monthly_sip_inflows.csv', '1788499984721-4b860901-05_category_inflows.csv', '1788499985036-da4a0c4a-06_industry_folio_count.csv', '1788499985420-bb134abf-07_scheme_performance.csv', '1788499984134-b0cbf625-03_aum_by_fund_house.csv']


In [ ]:
import pandas as pd
import os

raw_path = "/content/drive/MyDrive/Bluestock /capstone/raw"

df_transactions = pd.read_csv(os.path.join(raw_path, "1788499980509-304c1255-08_investor_transactions.csv"))
df_holdings = pd.read_csv(os.path.join(raw_path, "1788499982117-e3d6ab98-09_portfolio_holdings.csv"))
df_index = pd.read_csv(os.path.join(raw_path, "1788499982615-f9647ab2-10_benchmark_indices.csv"))
df_fund = pd.read_csv(os.path.join(raw_path, "1788499983024-b042c300-01_fund_master.csv"))
df_nav = pd.read_csv(os.path.join(raw_path, "1788499983331-4389156d-02_nav_history.csv"))
df_monthlySIP = pd.read_csv(os.path.join(raw_path, "1788499984405-d702a6c6-04_monthly_sip_inflows.csv"))
df_inflow = pd.read_csv(os.path.join(raw_path, "1788499984721-4b860901-05_category_inflows.csv"))
df_folios = pd.read_csv(os.path.join(raw_path, "1788499985036-da4a0c4a-06_industry_folio_count.csv"))
df_performance = pd.read_csv(os.path.join(raw_path, "1788499985420-bb134abf-07_scheme_performance.csv"))
df_aum = pd.read_csv(os.path.join(raw_path, "1788499984134-b0cbf625-03_aum_by_fund_house.csv"))

print("All 10 raw datasets loaded successfully!")

All 10 raw datasets loaded successfully!


In [ ]:
datasets = {
    "Investor Transactions": df_transactions,
    "Portfolio Holdings": df_holdings,
    "Benchmark Indices": df_index,
    "Fund Master": df_fund,
    "NAV History": df_nav,
    "Monthly SIP Inflows": df_monthlySIP,
    "Category Inflows": df_inflow,
    "Industry Folio Count": df_folios,
    "Scheme Performance": df_performance,
    "AUM by Fund House": df_aum
}

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print("\nShape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nFirst 5 Rows:")
    print(df.head())


INVESTOR TRANSACTIONS

Shape:
(32778, 13)

Data Types:
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object

First 5 Rows:
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0  

#4. Initial Data Quality Observations

Initial Data Quality Observations:

1. All 10 datasets were successfully loaded using Pandas.

2. Several date and month columns are currently stored as object data types and will require conversion to datetime format during the data cleaning stage.

3. The Monthly SIP Inflows dataset contains missing values in the yoy_growth_pct column for early records where previous-year comparison data is unavailable.

4. The datasets contain a combination of numerical, categorical, date, and identifier fields.

5. Dataset sizes vary significantly, with NAV History and Investor Transactions being the largest datasets.

6. Further validation and detailed cleaning will be performed during the data cleaning stage.

#5. Live NAV Data Fetching

In [ ]:
import requests
import pandas as pd

url = "https://api.mfapi.in/mf/125497"

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

print("Scheme Name:", data["meta"]["scheme_name"])

nav_data = pd.DataFrame(data["data"])

print(nav_data.head())

Status Code: 200
Scheme Name: SBI SMALL CAP FUND - Direct Plan - Growth
         date        nav
0  04-09-2026  216.65250
1  03-09-2026  216.40370
2  02-09-2026  216.08250
3  01-09-2026  216.74250
4  31-08-2026  217.61840


In [ ]:
# Save live NAV data as raw CSV

nav_data.to_csv(
    "live_nav_125497.csv",
    index=False
)

print("Live NAV data saved successfully as live_nav_125497.csv")

Live NAV data saved successfully as live_nav_125497.csv


#6. Fetch NAV for Key Mutual Fund Schemes

In [ ]:
import requests
import pandas as pd
import os

key_schemes = {
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_Large_Cap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

all_live_nav = {}

for scheme_name, scheme_code in key_schemes.items():

    url = f"https://api.mfapi.in/mf/{scheme_code}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        nav_df = pd.DataFrame(data["data"])

        # Add scheme information
        nav_df["scheme_code"] = scheme_code
        nav_df["scheme_name"] = data["meta"]["scheme_name"]

        # Save individual raw CSV
        file_name = f"live_nav_{scheme_code}.csv"
        nav_df.to_csv(file_name, index=False)

        all_live_nav[scheme_name] = nav_df

        print(f"✓ {scheme_name} loaded successfully")
        print(f"  API Scheme Name: {data['meta']['scheme_name']}")
        print(f"  Rows fetched: {len(nav_df)}\n")

    else:
        print(f"✗ Failed to fetch {scheme_name}")
        print("Status Code:", response.status_code)

✓ SBI_Bluechip loaded successfully
  API Scheme Name: Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment
  Rows fetched: 3303

✓ ICICI_Bluechip loaded successfully
  API Scheme Name: Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option
  Rows fetched: 3375

✓ Nippon_Large_Cap loaded successfully
  API Scheme Name: Nippon India Large Cap Fund - Direct Plan - Growth Option
  Rows fetched: 3366

✓ Axis_Bluechip loaded successfully
  API Scheme Name: HDFC Money Market Fund - Direct Plan - Growth Option
  Rows fetched: 3632

✓ Kotak_Bluechip loaded successfully
  API Scheme Name: Quant Mid Cap Fund - Direct Plan - Growth Option
  Rows fetched: 3369



In [ ]:
scheme_codes = [119551, 120503, 118632, 119092, 120841]

all_live_nav = {}

for scheme_code in scheme_codes:

    url = f"https://api.mfapi.in/mf/{scheme_code}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        nav_df = pd.DataFrame(data["data"])
        nav_df["amfi_code"] = scheme_code
        nav_df["scheme_name"] = data["meta"]["scheme_name"]

        file_name = f"live_nav_{scheme_code}.csv"
        nav_df.to_csv(file_name, index=False)

        all_live_nav[scheme_code] = nav_df

        print(f"✓ AMFI Code {scheme_code} loaded successfully")
        print(f"  Scheme Name: {data['meta']['scheme_name']}")
        print(f"  Rows fetched: {len(nav_df)}\n")

    else:
        print(f"✗ Failed to fetch AMFI Code {scheme_code}")

✓ AMFI Code 119551 loaded successfully
  Scheme Name: Aditya Birla Sun Life Banking & PSU Debt Fund - Direct Plan - IDCW-Re-investment
  Rows fetched: 3303

✓ AMFI Code 120503 loaded successfully
  Scheme Name: Axis ELSS- Tax Saver Fund - Direct Plan - Growth Option
  Rows fetched: 3375

✓ AMFI Code 118632 loaded successfully
  Scheme Name: Nippon India Large Cap Fund - Direct Plan - Growth Option
  Rows fetched: 3366

✓ AMFI Code 119092 loaded successfully
  Scheme Name: HDFC Money Market Fund - Direct Plan - Growth Option
  Rows fetched: 3632

✓ AMFI Code 120841 loaded successfully
  Scheme Name: Quant Mid Cap Fund - Direct Plan - Growth Option
  Rows fetched: 3369



#7. Fund Master Exploration

In [ ]:
print("UNIQUE FUND HOUSES")
print(df_fund["fund_house"].unique())

print("\n" + "=" * 50)
print("UNIQUE CATEGORIES")
print(df_fund["category"].unique())

print("\n" + "=" * 50)
print("UNIQUE SUB-CATEGORIES")
print(df_fund["sub_category"].unique())

print("\n" + "=" * 50)
print("UNIQUE RISK CATEGORIES")
print(df_fund["risk_category"].unique())

UNIQUE FUND HOUSES
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

UNIQUE CATEGORIES
['Equity' 'Debt']

UNIQUE SUB-CATEGORIES
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

UNIQUE RISK CATEGORIES
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [ ]:
print("UNIQUE RISK GRADES")
print(df_performance["risk_grade"].unique())

UNIQUE RISK GRADES
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


Fund Master Observations:
The fund master dataset contains 40 mutual fund schemes managed by 10 different fund houses.

The schemes are broadly classified into two main categories: Equity and Debt.

The dataset contains multiple sub-categories, including Large Cap, Small Cap, Mid Cap, Gilt, Short Duration, Value, Liquid, Index/ETF, Flexi Cap, Index, Large & Mid Cap, and ELSS.

The schemes are assigned risk categories ranging from Low to Very High.

Risk Grade Exploration: The Scheme Performance dataset contains five risk grades: Low, Moderate, Moderately High, High, and Very High.

#8. AMFI Code Validation and Data Quality Check

In [ ]:
fund_codes = set(df_fund["amfi_code"].unique())
nav_codes = set(df_nav["amfi_code"].unique())

missing_codes = fund_codes - nav_codes

print("Total AMFI codes in Fund Master:", len(fund_codes))
print("Total AMFI codes in NAV History:", len(nav_codes))
print("AMFI codes missing from NAV History:", len(missing_codes))

if len(missing_codes) == 0:
    print("\n✓ All AMFI codes in Fund Master exist in NAV History.")
else:
    print("\n✗ Missing AMFI codes:")
    print(missing_codes)

Total AMFI codes in Fund Master: 40
Total AMFI codes in NAV History: 40
AMFI codes missing from NAV History: 0

✓ All AMFI codes in Fund Master exist in NAV History.


AMFI Code Validation Summary:

All 40 unique AMFI codes present in the Fund Master dataset were successfully found in the NAV History dataset.

No AMFI codes were missing from NAV History.

This confirms that amfi_code can be used as a reliable key to connect the Fund Master and NAV History datasets.

Overall, the initial dataset validation was successful. Date and month columns stored as object data types will require conversion during the data cleaning stage.

#9. Create Data Ingestion Script

In [ ]:
data_ingestion_code = '''
import pandas as pd
import os

# Path to raw datasets
RAW_PATH = "/content/drive/MyDrive/Bluestock /capstone/raw"

# Dataset filenames
files = {
    "investor_transactions": "1788499980509-304c1255-08_investor_transactions.csv",
    "portfolio_holdings": "1788499982117-e3d6ab98-09_portfolio_holdings.csv",
    "benchmark_indices": "1788499982615-f9647ab2-10_benchmark_indices.csv",
    "fund_master": "1788499983024-b042c300-01_fund_master.csv",
    "nav_history": "1788499983331-4389156d-02_nav_history.csv",
    "monthly_sip_inflows": "1788499984405-d702a6c6-04_monthly_sip_inflows.csv",
    "category_inflows": "1788499984721-4b860901-05_category_inflows.csv",
    "industry_folio_count": "1788499985036-da4a0c4a-06_industry_folio_count.csv",
    "scheme_performance": "1788499985420-bb134abf-07_scheme_performance.csv",
    "aum_by_fund_house": "1788499984134-b0cbf625-03_aum_by_fund_house.csv"
}

# Load and inspect datasets
for dataset_name, file_name in files.items():
    file_path = os.path.join(RAW_PATH, file_name)
    df = pd.read_csv(file_path)

    print("\\n" + "=" * 60)
    print(dataset_name.upper())
    print("=" * 60)
    print("Shape:", df.shape)
    print("\\nData Types:")
    print(df.dtypes)
    print("\\nFirst 5 Rows:")
    print(df.head())

print("\\nAll 10 datasets loaded and inspected successfully!")
'''

with open("data_ingestion.py", "w") as file:
    file.write(data_ingestion_code)

print("data_ingestion.py created successfully!")

data_ingestion.py created successfully!


#10. Create Live NAV Fetch Script

In [ ]:
live_nav_code = '''
import requests
import pandas as pd

# Required AMFI scheme codes
scheme_codes = [
    125497,
    119551,
    120503,
    118632,
    119092,
    120841
]

for scheme_code in scheme_codes:

    url = f"https://api.mfapi.in/mf/{scheme_code}"
    response = requests.get(url)

    if response.status_code == 200:

        data = response.json()

        # Convert NAV data to DataFrame
        nav_df = pd.DataFrame(data["data"])

        # Add scheme information
        nav_df["amfi_code"] = scheme_code
        nav_df["scheme_name"] = data["meta"]["scheme_name"]

        # Save as CSV
        file_name = f"live_nav_{scheme_code}.csv"
        nav_df.to_csv(file_name, index=False)

        print(f"Successfully fetched AMFI Code: {scheme_code}")
        print(f"Scheme Name: {data['meta']['scheme_name']}")
        print(f"Rows fetched: {len(nav_df)}")
        print("-" * 50)

    else:
        print(f"Failed to fetch AMFI Code: {scheme_code}")
        print(f"Status Code: {response.status_code}")

print("Live NAV fetching completed!")
'''

with open("live_nav_fetch.py", "w") as file:
    file.write(live_nav_code)

print("live_nav_fetch.py created successfully!")

live_nav_fetch.py created successfully!
